# Colab setup: nr-jamming-localization

Run these cells top to bottom at the start of **every** Colab session. They clone (or update)
the repo on the Colab machine, install it, and set where generated data and results are saved.

**Two ways to run it**
- **VS Code** (Google Colab extension): open this notebook, *Select Kernel → Colab → New Colab
  Server*, pick a GPU. Leave `BROWSER_COLAB = False`.
- **Browser** ([colab.research.google.com](https://colab.research.google.com)): open it from
  GitHub, *Runtime → Change runtime type → GPU*, and set `BROWSER_COLAB = True`.

**One-time: GitHub token.** github.com → Settings → Developer settings → Personal access tokens →
*Fine-grained tokens* → *Generate new token*. Repository access: *Only select repositories* →
`nr-jamming-localization`. Permissions → *Contents*: **Read and write** (read-only is enough if
you never push from Colab). Keep it in your password manager. In VS Code you paste it when cell 2
asks. In the browser you can store it once as the Colab secret `GITHUB_TOKEN` (🔑 *Secrets* in
the left sidebar, with *Notebook access* on).

In [ ]:
# 1. Settings and GPU check
# False: VS Code Colab extension. Colab Secrets and Drive mounting don't work there yet
#        (googlecolab/colab-vscode#215), so the token is asked for and files stay on the
#        Colab machine. This also works in the browser, it's just less convenient.
# True:  colab.research.google.com. Uses the GITHUB_TOKEN secret and saves to Google Drive.
BROWSER_COLAB = False

!nvidia-smi -L || echo "No GPU attached: choose a GPU runtime"

In [ ]:
# 2. Clone the repo, or pull the latest version if it is already here
import os
import subprocess
from getpass import getpass

REPO = "erhankarakoca/nr-jamming-localization"
ROOT = "/content/nr-jamming-localization"


def git(*args):
    r = subprocess.run(["git", *args], check=False, capture_output=True, text=True)
    print(r.stdout + r.stderr, end="")
    r.check_returncode()


# Git reads the token from the environment, so it never appears in a URL or in cell output.
if not os.environ.get("GITHUB_TOKEN"):
    if BROWSER_COLAB:
        from google.colab import userdata

        os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    else:
        os.environ["GITHUB_TOKEN"] = getpass("GitHub token: ")
git("config", "--global", "credential.helper",
    '!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f')
git("config", "--global", "user.name", "Erhan Karakoca")
git("config", "--global", "user.email", "64256175+erhankarakoca@users.noreply.github.com")

if os.path.isdir(f"{ROOT}/.git"):
    git("-C", ROOT, "pull", "--ff-only")
else:
    git("clone", f"https://github.com/{REPO}.git", ROOT)
os.chdir(ROOT)
git("log", "--oneline", "-3")

In [ ]:
# 3. Install the project into Colab's Python (keeps Colab's CUDA build of PyTorch)
!pip install -q -e . pytest

import sys

sys.path.insert(0, f"{ROOT}/src")  # an editable install is otherwise only seen after a restart

from importlib.metadata import version

import numpy
import torch

stale = [m.__name__ for m in (numpy, torch)
         if version(m.__name__).split("+")[0] != m.__version__.split("+")[0]]
if stale:
    print(f"pip upgraded {stale}: restart the session/kernel, then run all cells again.")
else:
    import sionna.phy

    print(f"torch {torch.__version__} | sionna {version('sionna-no-rt')} "
          f"| device {sionna.phy.config.device}")

# autoreload picks up code changes after a `git pull` without restarting. On some Colab images its
# IPython extension fails to import under Python 3.13; then restart the kernel after pulling instead.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload is not available here: restart the kernel after pulling code changes.")

In [ ]:
# 4. Where generated data and results go
if BROWSER_COLAB:
    # Google Drive keeps them after the session ends.
    from google.colab import drive

    drive.mount("/content/drive")
    STORE = "/content/drive/MyDrive/nr-jamming-localization"
    os.environ["JAMLOC_DATA_DIR"] = f"{STORE}/data"
    os.environ["JAMLOC_RESULTS_DIR"] = f"{STORE}/results"
else:
    # data/ and results/ inside the repo on the Colab machine. They are deleted when the
    # session ends: datasets can be regenerated, but download results first.
    os.environ.pop("JAMLOC_DATA_DIR", None)
    os.environ.pop("JAMLOC_RESULTS_DIR", None)

from jamloc import paths

print("data    ->", paths.data_dir())
print("results ->", paths.results_dir())

In [ ]:
# 5. Smoke test
!python -m pytest -q

## Running the experiment

Cell 6 regenerates the datasets, trains the four U-Net variants with three seeds each, evaluates
them (test set, scheduling-aware jammer, and two SNR ranges), and packs everything into
`results/full_run.zip`. On a Colab GPU this takes roughly 15-30 minutes. Progress is printed as it
goes; if the session drops, re-run cells 1-4 and then cell 6 with `--reuse-data` to skip the
datasets.

Download `results/full_run.zip` when it finishes (see *Working from Colab* below), unzip it into
`results/` on your computer, and the evaluation figures and trained models are there.

In [ ]:
# 6. Whole experiment -> results/full_run.zip
!python -m jamloc.experiments.full_run --seeds 0 1 2

## Working from Colab

- **Code changes:** edit on your computer, `git push`, then re-run cell 2 here. `autoreload`
  picks up the changed modules without a restart (if cell 3 says it is not available, restart
  the kernel and run cells 1-4 again).
- **Saving results (VS Code):** before the session ends, download what you need from
  `/content/nr-jamming-localization/results` in the Colab extension's remote file view. Keep
  results small (metrics, figures, checkpoints); very large downloads can fail.
- **Saving results (browser):** they are already on Google Drive.
- **Editing here instead:** commit and push before the session ends with
  `!git add -A && git commit -m "..." && git push`.